## item def

This notebook introduces `item def`; after running it you can declare named item types that flow between actions, giving the action sequence a typed material vocabulary.

The `ApplyHeat` action from the previous notebook transforms energy — but what physical things move through the toaster? `item def` in SysML v2 names the typed flows: the bread entering, the toast exiting, and the signal that cancels the cycle. Items are not parts (they do not own sub-structure); they are the typed goods that actions produce and consume. This notebook adds `Start`, `Finish`, and `Cancel` item definitions.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
}"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: an item def that specializes an undefined type
# raises "unresolved reference" at the specialization site.
bad_source = """
package Bad {
    item def BadItem :> UndefinedBase;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
for name in ("ToasterDemo::Start", "ToasterDemo::Finish", "ToasterDemo::Cancel"):
    sym = model.find(name)
    assert sym is not None, f"Not found: {name}"
    print(f"{sym.id}: kind={sym.kind}")
conn.close()

`item def Start; item def Finish; item def Cancel;` (A-F) are parsed and registered by OpenSysML (O-S); `model.find()` retrieves each item symbol with its kind (E).

Try the chapter exercise in `exercises/ch04/exercise.ipynb`: add `item def CoffeeGrounds` and `item def BrewedCoffee` to your coffee maker model and confirm both are findable.